# M1 — SCKN bestseller labels

**Question:** What does the SCKN chart data look like, and how many unique books ever appeared in the top 10?

**Deliverable:** `data/raw/sckn_charts.csv` — every chart entry from 2003 to present.

**Label rule (to be applied in M4 after NKC matching):**  
`label = 1` if the book appears at `rank ≤ 10` in **any** SCKN category (fiction, non-fiction, children's) in any week.  
All three categories are in scope — the thesis goal is category-agnostic, and `category` will be used as a model feature.

**ISBN usage:**  
The ISBN in this chart is the **Czech edition ISBN**. It is used only for the SCKN → NKC hop (both sides hold the Czech edition ISBN, so it can be an exact join key). It cannot be used for the NKC → Goodreads hop, which requires the original-language ISBN — that hop uses OCLC or original title+author instead.

In [2]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

OUTPUT = ROOT / 'data' / 'raw' / 'sckn_charts.csv'

## 1. Scrape (skip if CSV already exists)

In [3]:
if OUTPUT.exists():
    print(f'Already scraped → {OUTPUT}  ({OUTPUT.stat().st_size / 1024:.0f} KB)')
else:
    from src.data.scrape_sckn import scrape_all, save_csv
    print('Starting scrape — this takes ~15 minutes at 0.5 s/request ...')
    rows = scrape_all(start_year=2003)
    save_csv(rows, OUTPUT)
    print(f'Saved {len(rows):,} rows → {OUTPUT}')

Starting scrape — this takes ~15 minutes at 0.5 s/request ...
2003: 1642 entries


2004: 1739 entries
2005: 1578 entries
2006: 1569 entries
2007: 1575 entries


2008: 1530 entries
2009: 1590 entries
2010: 1560 entries
2011: 1560 entries
2012: 1540 entries
2013: 1560 entries
2014: 1562 entries
2015: 1593 entries


2016: 1530 entries


2017: 1530 entries
2018: 1530 entries
2019: 1500 entries
2020: 1200 entries


2021: 1290 entries
2022: 1560 entries
2023: 1560 entries
2024: 1560 entries
2025: 1560 entries
2026: 510 entries
Saved 35,928 rows → /home/firstone/Bachelors-thesis/data/raw/sckn_charts.csv


## 2. Load and preview

In [4]:
df = pd.read_csv(OUTPUT, dtype={'isbn': str})
print(f'Shape: {df.shape}')
df.head(10)

Shape: (35928, 8)


,year,week,category,rank,isbn,author,title,publisher
0,2003,1,beletrie,1,80-86718-00-X,"Růičková Helena, Formáčková Marie",Deník mezi ivotem a smrtí,Formát
1,2003,1,beletrie,2,80-86631-02-8,Sommerová Olga,O čem sní eny 2,Slávka Kopecká
2,2003,1,beletrie,3,80-238-5878-5,Růičková Helena,A tak jsem la ivotem,Metramedia
3,2003,1,beletrie,4,80-200-1023-8,Branald Adolf,Převleky mého města,Academia
4,2003,1,beletrie,5,80-7203-410-3,Coelho Paulo,Poutník - Mágův deník,Argo
5,2003,1,beletrie,6,80-204-0983-1,Tolkien J. R. R.,Hobit aneb Cesta tam a zase zpátky (ilustr.vyd.),Mladá fronta
6,2003,1,beletrie,7,80-200-1038-6,Lodge David,Profesorské hrátky,Academia
7,2003,1,beletrie,8,80-7203-375-1,Saroyan William,Tracyho tygr / Tracy´s Tygr,Argo
8,2003,1,beletrie,9,80-7203-438-3,Adams Douglas,Stopařův průvodce Galaxií,Argo
9,2003,1,beletrie,10,80-7303-093-4,Deaver Jeffery,Kamenná opice,Domino


In [5]:
print('Dtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())

Dtypes:
year         int64
week         int64
category       str
rank         int64
isbn           str
author         str
title          str
publisher      str
dtype: object

Null counts:
year            0
week            0
category        0
rank            0
isbn           14
author       2193
title          18
publisher      25
dtype: int64


## 3. Year and category coverage

In [6]:
rename = {
    'beletrie': 'fiction',
    'naučná literatura': 'nonfiction',
    'literatura pro děti a mládež': 'children',
}

year_cat = (
    df.groupby(['year', 'category'])
    .size()
    .unstack(fill_value=0)
    .rename(columns=rename)
)
year_cat['total'] = year_cat.sum(axis=1)
year_cat

category,fiction,children,nonfiction,total
year,,,,
2003,529,572,541,1642
2004,565,600,574,1739
2005,524,532,522,1578
2006,522,526,521,1569
2007,526,525,524,1575
2008,510,510,510,1530
2009,530,530,530,1590
2010,520,520,520,1560
2011,520,520,520,1560


## 4. Unique books that ever appeared in any top 10

In [7]:
# All categories, rank ≤ 10 — this is the label = 1 candidate pool
top10 = df[df['rank'] <= 10].copy()

print(f'Total top-10 entries (rows):      {len(top10):,}')
print(f'Unique ISBNs in top-10:           {top10["isbn"].nunique():,}')
print(f'Unique (title, author) pairs:     {top10.drop_duplicates(["title","author"]).shape[0]:,}')
print()
print('Breakdown by category:')
print(top10.groupby('category')[['isbn']].nunique().rename(columns={'isbn': 'unique_isbns'}).rename(index=rename))

Total top-10 entries (rows):      35,611
Unique ISBNs in top-10:           8,249
Unique (title, author) pairs:     9,794

Breakdown by category:
            unique_isbns
category                
fiction             2551
children            2830
nonfiction          3000


In [8]:
# Most-charted books overall (appeared in top-10 most weeks, any category)
top_books = (
    top10
    .groupby(['isbn', 'author', 'title', 'category'])
    .size()
    .rename('weeks_in_top10')
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
top_books['category'] = top_books['category'].map(rename)
top_books

,isbn,author,title,category,weeks_in_top10
0,978-80-7555-097-2,James Clear,Atomové návyky,nonfiction,177
1,978-80-00-02225-3,Kinney Jeff,Deník malého poseroutky,children,155
2,80-86518-62-0,Brown Dan,ifra mistra Leonarda,fiction,108
3,978-80-88362-00-5,Karin Lednická,Šikmý kostel,fiction,99
4,978-80-7491-940-4,Alena Mornštajnová,Hana,fiction,99
5,978-80-7447-737-9,Smithová Keri,Destrukční deník,children,97
6,978-80-7432-910-4,Vojtěch Matocha,Prašina,children,93
7,978-80-00-01462-3,Saint-Exupéry Antoine de,Malý princ,children,90
8,978-80-903944-3-8,Sedláček Tomáš,Ekonomie dobra a zla,nonfiction,82
9,978-80-969686-9-5,Mačingová Antónia,Zhubněte jednou provždy,nonfiction,81


## 5. ISBN quality check

In [9]:
# Czech edition ISBNs — used for SCKN → NKC exact join in M4
isbns = df['isbn'].dropna().str.replace('-', '', regex=False)

print('ISBN length distribution:')
print(isbns.str.len().value_counts().sort_index())

print(f'\nStarts with 978: {(isbns.str.startswith("978")).sum():,}')
print(f'Starts with 979: {(isbns.str.startswith("979")).sum():,}')
print(f'Empty/missing:   {df["isbn"].isna().sum() + (df["isbn"] == "").sum():,}')

ISBN length distribution:
isbn
10     7452
11        8
12      648
13    27805
42        1
Name: count, dtype: int64

Starts with 978: 28,441
Starts with 979: 0
Empty/missing:   14


## 6. Summary

**Key findings to fill in after running:**
- Chart data: 2003–present, ~52 weeks × 30 entries/week = ~1,560 entries/year
- `label = 1` pool: unique (title, author) pairs that appeared at rank ≤ 10 in any category
- ISBNs are predominantly ISBN-13 (13 digits, starts with 978) — reliable key for SCKN → NKC join

**Join strategy clarified:**
- SCKN → NKC: Czech ISBN (`020 $a`) exact match, fuzzy title+author fallback
- NKC → Goodreads: OCLC (`035 $a`) or original title+author (`240 $a` + `100 $a`) — Czech ISBN is useless here

**Next step (M2):** Parse the NKC XML to extract Czech translations with their OCLC numbers, original titles, and Czech publication years.